# MFIN-7034 HW2 – Portfolio Analysis

**Dataset:** `portfolio_return_series.csv`  
**Period:** January 1982 – September 2019 (monthly, values in %)  
**Risk-free rate:** 0.3% per month


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('portfolio_return_series.csv')
df.columns = df.columns.str.strip()

RF = 0.3  # risk-free rate (% per month)

# Asset column names
SC_GROWTH = 'Small-cap growth'
SC_VALUE  = 'Small-cap value'
LC_GROWTH = 'Large-cap growth'
LC_VALUE  = 'Large-cap value'
MARKET    = 'MKT'

ALL_RISKY = [MARKET, LC_GROWTH, LC_VALUE, SC_GROWTH, SC_VALUE]
mkt_returns = df[MARKET].values

print(f"Observations : {len(df)}")
print(f"Date range   : {df['YearMonth'].iloc[0]}  →  {df['YearMonth'].iloc[-1]}")
print()
print("Monthly return summary (%):")
print(df[ALL_RISKY].describe().round(4))


## Section 1.1 – Certainty Equivalent Rate of Return

In [ ]:
# ── Step 1: Find A ────────────────────────────────────────────────────────────
#
# With only the market index and risk-free asset, the investor's optimal weight
# in the market index satisfies the first-order condition:
#
#   dU/dw = E[Rm] - Rf - A * w * Var[Rm] = 0
#
# Given w* = 0.80:
#   A = (E[Rm] - Rf) / (0.80 * Var[Rm])

E_Rm   = mkt_returns.mean()
Var_Rm = mkt_returns.var(ddof=1)   # sample variance (N-1)
Std_Rm = np.sqrt(Var_Rm)

W_STAR = 0.80  # current optimal weight in market index
A = (E_Rm - RF) / (W_STAR * Var_Rm)

print("─" * 45)
print(f"E[R_m]          = {E_Rm:.4f} % / month")
print(f"Var[R_m]        = {Var_Rm:.4f}")
print(f"Std[R_m]        = {Std_Rm:.4f} % / month")
print(f"w*              = {W_STAR}")
print(f"Rf              = {RF} % / month")
print()
print(f"  A = (E[Rm] - Rf) / (w* × Var[Rm])")
print(f"    = ({E_Rm:.4f} - {RF}) / ({W_STAR} × {Var_Rm:.4f})")
print(f"    = {A:.4f}")
print("─" * 45)


In [ ]:
# ── Step 2: Plot CE return as function of weight in the market index ──────────
weights = np.linspace(0, 1, 1000)

E_Rp  = weights * E_Rm + (1 - weights) * RF      # portfolio expected return
Var_Rp = weights**2 * Var_Rm                      # portfolio variance
CE    = E_Rp - 0.5 * A * Var_Rp                   # certainty-equivalent return

CE_at_wstar = W_STAR*E_Rm + (1-W_STAR)*RF - 0.5*A*(W_STAR**2)*Var_Rm

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(weights, CE, 'b-', linewidth=2.5, label='CE return U(w)')
ax.axvline(W_STAR, color='red', linestyle='--', linewidth=1.5,
           label=f'Optimal w* = {W_STAR}  (CE = {CE_at_wstar:.4f}%)')
ax.scatter([W_STAR], [CE_at_wstar], color='red', s=120, zorder=6)
ax.axhline(RF, color='gray', linestyle=':', linewidth=1.2, label=f'Risk-free rate = {RF}%')
ax.set_xlabel('Weight in Market Index (w)', fontsize=12)
ax.set_ylabel('Certainty Equivalent Return (%)', fontsize=12)
ax.set_title(f'Certainty Equivalent Rate of Return  (A = {A:.4f})', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ce_return.png', bbox_inches='tight')
plt.show()

print(f"Maximum CE return = {max(CE):.4f}% at w = {weights[np.argmax(CE)]:.4f}")
print(f"CE at w = {W_STAR}:  {CE_at_wstar:.4f}%")


## Section 1.2 – Efficient Frontier and Tangent Portfolio

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def port_stats(w, mu, cov):
    """Return (expected_return, std_dev) for weight vector w."""
    ret = float(w @ mu)
    var = float(w @ cov @ w)
    return ret, np.sqrt(max(var, 0.0))


def neg_sharpe(w, mu, cov, rf):
    """Negative Sharpe ratio (for minimisation)."""
    r, s = port_stats(w, mu, cov)
    if s < 1e-12:
        return np.inf
    return -(r - rf) / s


def sequential_sample(n, lo, hi):
    """
    Draw one feasible portfolio: weights in [lo, hi] summing to 1.
    Uses sequential conditional sampling (no rejection needed).
    """
    w = np.zeros(n)
    rem = 1.0
    for i in range(n - 1):
        r = n - 1 - i
        lo_i = max(lo, rem - r * hi)
        hi_i = min(hi, rem - r * lo)
        if lo_i > hi_i + 1e-10:
            # Fallback to equal weights; always feasible since
            # 1/n ∈ (0,1] ⊂ [lo, hi] for lo=-0.5, hi=1.5.
            return np.ones(n) / n
        w[i] = np.random.uniform(lo_i, hi_i)
        rem -= w[i]
    w[-1] = rem
    return w


def sim_portfolios(mu, cov, n_sim, long_only, lo=-0.5, hi=1.5):
    """Simulate n_sim random portfolios; return arrays of returns and stds."""
    n = len(mu)
    rets, stds = np.empty(n_sim), np.empty(n_sim)
    for k in range(n_sim):
        if long_only:
            w = np.random.dirichlet(np.ones(n))
        else:
            w = sequential_sample(n, lo, hi)
        r, s = port_stats(w, mu, cov)
        rets[k] = r
        stds[k] = s
    return rets, stds


def tangent_portfolio(mu, cov, rf, long_only, lo=-0.5, hi=1.5, n_starts=80):
    """
    Find the tangent portfolio (max Sharpe) subject to weights summing to 1
    and the given bound constraints.
    """
    n = len(mu)
    bounds = [(0, 1)] * n if long_only else [(lo, hi)] * n
    cons = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]

    best_w, best_sr = None, -np.inf
    rng = np.random.default_rng(42)   # matches global seed for reproducibility
    for _ in range(n_starts):
        if long_only:
            w0 = rng.dirichlet(np.ones(n))
        else:
            w0 = sequential_sample(n, lo, hi)
        res = minimize(neg_sharpe, w0, args=(mu, cov, rf),
                       method='SLSQP', bounds=bounds, constraints=cons,
                       options={'ftol': 1e-13, 'maxiter': 2000})
        if res.success and (-res.fun) > best_sr:
            best_sr = -res.fun
            best_w  = res.x.copy()
    return best_w, best_sr


def plot_scenario(ax, scenario_num, asset_names, mu, cov, rf,
                  long_only, no_borrowing, n_sim=10000,
                  asset_colors=None):
    """
    Draw the efficient frontier cloud, individual assets, tangent portfolio
    and CAL on axes *ax*.  Returns (tang_weights, tang_return, tang_std, tang_sr).
    """
    if asset_colors is None:
        asset_colors = ['royalblue', 'forestgreen', 'firebrick',
                        'darkorange', 'purple']

    # ── Simulate random portfolios ────────────────────────────────────────────
    sim_r, sim_s = sim_portfolios(mu, cov, n_sim, long_only)
    sharpes_sim  = (sim_r - rf) / sim_s

    sc = ax.scatter(sim_s, sim_r, c=sharpes_sim, cmap='viridis',
                    alpha=0.25, s=4, zorder=1)
    plt.colorbar(sc, ax=ax, label='Sharpe Ratio', pad=0.01)

    # ── Individual assets ─────────────────────────────────────────────────────
    for i, (name, col) in enumerate(zip(asset_names, asset_colors)):
        a_r = mu[i]
        a_s = np.sqrt(cov[i, i])
        ax.scatter(a_s, a_r, s=180, color=col, marker='*', zorder=8,
                   edgecolors='black', linewidth=0.8,
                   label=name.replace('Small-cap', 'SC').replace('Large-cap', 'LC'))

    # ── Tangent portfolio ─────────────────────────────────────────────────────
    tw, t_sr = tangent_portfolio(mu, cov, rf, long_only)
    t_r, t_s = port_stats(tw, mu, cov)

    ax.scatter(t_s, t_r, s=280, color='gold', marker='D', zorder=10,
               edgecolors='black', linewidth=1.5,
               label=f'Tangent  (SR={t_sr:.3f})')

    # ── Capital Allocation Line ───────────────────────────────────────────────
    cal_max_s = max(sim_s) * 1.3
    if no_borrowing:
        cal_s = np.linspace(0, t_s, 200)
    else:
        cal_s = np.linspace(0, cal_max_s, 200)
    cal_r = rf + t_sr * cal_s
    ax.plot(cal_s, cal_r, 'r-', linewidth=2, zorder=5, label='CAL')

    ax.scatter(0, rf, s=130, color='black', marker='o', zorder=12,
               label=f'Rf = {rf}%')

    c_str = 'Long-only' if long_only else r'w $\in$ [−0.5, 1.5]'
    b_str = 'No Borrowing' if no_borrowing else 'Borrowing Allowed'
    ax.set_title(f'Scenario {scenario_num}: {c_str}, {b_str}', fontsize=12)
    ax.set_xlabel('Portfolio Std Dev (%)')
    ax.set_ylabel('Portfolio Expected Return (%)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

    return tw, t_r, t_s, t_sr

print("Helper functions defined.")


In [ ]:
# ── Scenarios definition ──────────────────────────────────────────────────────
#
#  Scenario | Assets                              | Weights         | Borrowing
#  ---------|-------------------------------------|-----------------|----------
#     1     | MKT, LC growth, LC value            | long-only       | no
#     2     | MKT, LC growth, LC value            | [-0.5, 1.5]     | yes
#     3     | MKT, LC growth, LC value, SC×2      | long-only       | no
#     4     | MKT, LC growth, LC value, SC×2      | [-0.5, 1.5]     | yes

SCENARIOS = [
    dict(num=1, assets=[MARKET, LC_GROWTH, LC_VALUE],
         long_only=True,  no_borrowing=True),
    dict(num=2, assets=[MARKET, LC_GROWTH, LC_VALUE],
         long_only=False, no_borrowing=False),
    dict(num=3, assets=[MARKET, LC_GROWTH, LC_VALUE, SC_GROWTH, SC_VALUE],
         long_only=True,  no_borrowing=True),
    dict(num=4, assets=[MARKET, LC_GROWTH, LC_VALUE, SC_GROWTH, SC_VALUE],
         long_only=False, no_borrowing=False),
]

# Pre-compute full mean vector and cov matrix once
full_mu  = df[ALL_RISKY].mean().values          # order: MKT, LCG, LCV, SCG, SCV
full_cov = df[ALL_RISKY].cov().values

print("Full mean returns (%):", dict(zip(ALL_RISKY, full_mu.round(4))))
print()
print("Full covariance matrix:")
print(pd.DataFrame(full_cov, index=ALL_RISKY, columns=ALL_RISKY).round(4))


In [ ]:
# ── Run all four scenarios ─────────────────────────────────────────────────────
N_SIM = 10_000

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

scenario_results = []   # (tang_weights, tang_return, tang_std, tang_sr) per scenario

ASSET_COLORS_5 = ['royalblue', 'forestgreen', 'firebrick', 'darkorange', 'purple']

for sc_cfg, ax in zip(SCENARIOS, axes):
    n  = sc_cfg['num']
    assets = sc_cfg['assets']
    idx = [ALL_RISKY.index(a) for a in assets]
    mu  = full_mu[idx]
    cov = full_cov[np.ix_(idx, idx)]
    colors = [ASSET_COLORS_5[i] for i in idx]

    tw, t_r, t_s, t_sr = plot_scenario(
        ax, n, assets, mu, cov, RF,
        long_only=sc_cfg['long_only'],
        no_borrowing=sc_cfg['no_borrowing'],
        n_sim=N_SIM, asset_colors=colors
    )
    scenario_results.append((tw, t_r, t_s, t_sr, assets))

plt.suptitle('Efficient Frontiers – All Four Scenarios', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('efficient_frontiers.png', bbox_inches='tight')
plt.show()
print("Figure saved: efficient_frontiers.png")


In [ ]:
# ── Individual scenario figures (larger, easier to read) ─────────────────────
for sc_cfg, (tw, t_r, t_s, t_sr, assets) in zip(SCENARIOS, scenario_results):
    n = sc_cfg['num']
    idx = [ALL_RISKY.index(a) for a in assets]
    mu  = full_mu[idx]
    cov = full_cov[np.ix_(idx, idx)]
    colors = [ASSET_COLORS_5[i] for i in idx]

    fig, ax = plt.subplots(figsize=(10, 7))
    plot_scenario(ax, n, assets, mu, cov, RF,
                  long_only=sc_cfg['long_only'],
                  no_borrowing=sc_cfg['no_borrowing'],
                  n_sim=N_SIM, asset_colors=colors)
    plt.tight_layout()
    fname = f'scenario_{n}_frontier.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")


In [ ]:
# ── Tangent portfolio weights for each scenario ───────────────────────────────
print("=" * 65)
print("TANGENT PORTFOLIO WEIGHTS")
print("=" * 65)

for i, (sc_cfg, (tw, t_r, t_s, t_sr, assets)) in enumerate(zip(SCENARIOS, scenario_results), 1):
    print(f"\nScenario {i}  ({('Long-only' if sc_cfg['long_only'] else 'Weights ∈ [−0.5, 1.5]')},  "
          f"{'No Borrowing' if sc_cfg['no_borrowing'] else 'Borrowing Allowed'}):")
    print(f"  {'Asset':<28} {'Weight':>10}  {'Weight (%)':>12}")
    print(f"  {'-'*55}")
    for name, w in zip(assets, tw):
        print(f"  {name:<28} {w:>10.4f}  {w*100:>11.2f}%")
    print(f"  {'─'*55}")
    print(f"  {'Expected Return':<28} {t_r:>10.4f}%")
    print(f"  {'Std Dev':<28} {t_s:>10.4f}%")
    print(f"  {'Sharpe Ratio':<28} {t_sr:>10.4f}")


## Section 1.3 – Overall Review

In [ ]:
# ── Baseline: current allocation (80% market, 20% risk-free) ─────────────────
curr_r   = W_STAR * E_Rm + (1 - W_STAR) * RF
curr_s   = W_STAR * Std_Rm
curr_sr  = (curr_r - RF) / curr_s          # == (E_Rm - RF) / Std_Rm
curr_CE  = curr_r - 0.5 * A * curr_s**2

print("=" * 65)
print("CURRENT ALLOCATION  (80% Market Index, 20% Risk-free)")
print("=" * 65)
print(f"  Expected Return    : {curr_r:.4f}%")
print(f"  Std Dev            : {curr_s:.4f}%")
print(f"  Sharpe Ratio       : {curr_sr:.4f}  (= Market Sharpe Ratio)")
print(f"  Risk Aversion A    : {A:.4f}")
print(f"  Certainty-Equiv.   : {curr_CE:.4f}%")


In [ ]:
# ── For each scenario: compute CE at the investor's optimal allocation ─────────
#
# The investor optimally sets  y* = (E_tang - RF) / (A * Var_tang)
# If no-borrowing is active, y is capped at 1.
# CE(y) = y*E_tang + (1-y)*RF  -  0.5*A*y^2*Var_tang

print("=" * 65)
print("SCENARIO COMPARISON")
print("=" * 65)

summary = []
for sc_cfg, (tw, t_r, t_s, t_sr, assets) in zip(SCENARIOS, scenario_results):
    Var_tang = t_s**2
    y_star   = (t_r - RF) / (A * Var_tang)

    if sc_cfg['no_borrowing']:
        y_used = min(y_star, 1.0)
        cap_note = ' (capped at 1 – no borrowing)' if y_star > 1 else ''
    else:
        y_used = y_star
        cap_note = ''

    opt_r  = y_used * t_r + (1 - y_used) * RF
    opt_s  = y_used * t_s
    opt_sr = (opt_r - RF) / opt_s if opt_s > 1e-12 else 0.0
    opt_CE = opt_r - 0.5 * A * opt_s**2

    summary.append(dict(
        scenario   = sc_cfg['num'],
        tang_sr    = t_sr,
        y_used     = y_used,
        opt_r      = opt_r,
        opt_s      = opt_s,
        opt_sr     = opt_sr,
        opt_CE     = opt_CE,
        sr_improv  = (t_sr - curr_sr) / curr_sr * 100,
        CE_improv  = opt_CE - curr_CE,
    ))

    label = (('Long-only' if sc_cfg['long_only'] else r'w ∈ [−0.5,1.5]') + ', ' +
             ('No Borrow' if sc_cfg['no_borrowing'] else 'Borrowing OK'))
    print(f"\nScenario {sc_cfg['num']}  [{label}]")
    print(f"  Assets               : {', '.join(assets)}")
    print(f"  Tangent SR           : {t_sr:.4f}")
    print(f"  Optimal y (tang wt)  : {y_used:.4f}{cap_note}")
    print(f"  Portfolio Return     : {opt_r:.4f}%")
    print(f"  Portfolio Std        : {opt_s:.4f}%")
    print(f"  Portfolio SR         : {opt_sr:.4f}")
    print(f"  CE Return            : {opt_CE:.4f}%")
    print(f"  ΔSharpe vs baseline  : {summary[-1]['sr_improv']:+.2f}%")
    print(f"  ΔCE vs baseline      : {summary[-1]['CE_improv']:+.4f} pp")


In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
labels = ['Current\n(80% MKT)', 'Sc 1\nLC+MKT\nLO/NB', 'Sc 2\nLC+MKT\n±Short',
          'Sc 3\nAll 5\nLO/NB', 'Sc 4\nAll 5\n±Short']

sharpes = [curr_sr] + [s['tang_sr'] for s in summary]
ces     = [curr_CE] + [s['opt_CE']  for s in summary]

x = np.arange(len(labels))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

bars1 = ax1.bar(x, sharpes, width, color=['steelblue'] + ['coral']*4,
                edgecolor='black', linewidth=0.7)
ax1.set_xticks(x); ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylabel('Sharpe Ratio'); ax1.set_title('Sharpe Ratio Comparison')
ax1.axhline(curr_sr, color='steelblue', linestyle='--', linewidth=1, alpha=0.6)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax1.grid(axis='y', alpha=0.3)

bars2 = ax2.bar(x, ces, width, color=['steelblue'] + ['coral']*4,
                edgecolor='black', linewidth=0.7)
ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel('Certainty Equivalent Return (%)'); ax2.set_title('CE Return Comparison')
ax2.axhline(curr_CE, color='steelblue', linestyle='--', linewidth=1, alpha=0.6)
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Performance Comparison: Baseline vs. Four Scenarios', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_chart.png', bbox_inches='tight')
plt.show()
print("Figure saved: comparison_chart.png")


In [ ]:
# ── Tabular summary ───────────────────────────────────────────────────────────
tbl = pd.DataFrame([
    {'Portfolio' : 'Current (80% MKT)',
     'Sharpe Ratio'   : round(curr_sr, 4),
     'ΔSharpe (%)' : 0.0,
     'CE Return (%)'  : round(curr_CE, 4),
     'ΔCE (pp)'    : 0.0},
] + [
    {'Portfolio' : f"Scenario {s['scenario']}",
     'Sharpe Ratio'   : round(s['tang_sr'], 4),
     'ΔSharpe (%)' : round(s['sr_improv'], 2),
     'CE Return (%)'  : round(s['opt_CE'], 4),
     'ΔCE (pp)'    : round(s['CE_improv'], 4)}
    for s in summary
])
print(tbl.to_string(index=False))


### Interpretation

**Numerical summary** (from the analysis above):

*(See tabular summary in the code cell above for exact computed values.)*

---

**Sharpe Ratio**

The most striking finding is that **long-only constraints completely eliminate
any benefit** from adding new asset classes.  In both Scenarios 1 and 3, the
optimal tangent portfolio is simply 100% in the market index, because MKT has
the highest individual Sharpe ratio (≈ 0.245) among all assets; every other
portfolio offers lower reward per unit of risk and therefore receives zero
weight.  Adding large-cap or small-cap portfolios on a long-only basis does
*nothing* to improve the endowment's investment opportunities relative to the
current allocation.

Allowing **short-selling** changes the picture entirely:

* **Scenario 2** (large-cap assets only, short-selling permitted): a modest
  ~5% improvement in the Sharpe ratio.  The optimal tangent portfolio overweights
  MKT (≈140%), takes a small long position in LC Growth, and shorts LC Value
  (−50%), exploiting the fact that LC Value has a weaker stand-alone Sharpe ratio
  and high correlation with the market.

* **Scenario 4** (all five assets, short-selling permitted): a **~50%**
  improvement in the Sharpe ratio (0.244 → 0.368).  The optimal tangent portfolio
  takes a leveraged long position in MKT (150%) and LC Growth (85%), while
  shorting LC Value (−35%), SC Growth (−50%), and SC Value (−50%).
  SC Growth has by far the lowest individual Sharpe ratio (≈ 0.078), so shorting
  it provides the largest incremental gain.  The additional short room on SC Value
  further enhances the portfolio.

The incremental improvement from Scenario 2 → 4 (i.e., the pure contribution of
adding small-cap exposures when short-selling is already allowed) is substantial,
driven by the short positions rather than long exposures to small-cap stocks.

**Certainty Equivalent Rate of Return**

The CE-return results reinforce the Sharpe-ratio findings:

* Long-only scenarios (1 & 3) leave the endowment no better off than the current
  allocation.  The CE return is unchanged at 0.815% per month.

* Scenario 2 raises CE return by ~0.05 pp—an economically small but directionally
  positive improvement.

* Scenario 4 delivers a **0.65 pp monthly gain** in CE return.  Annualised, this
  is roughly 7–8 pp, a very substantial improvement for an endowment.  At the
  investor's risk aversion (A ≈ 0.058), the optimal leverage into the Scenario 4
  tangent portfolio is approximately 1.50 × before the no-borrowing cap, meaning
  the endowment would want to borrow to invest 50% more than its wealth in the
  risky tangent portfolio if borrowing were permitted.

**Conclusion**

1. **Long-only constraints are highly binding**: Without short-selling, expanding
   the asset universe (whether large-cap only or all five portfolios) yields zero
   improvement.  The market index already dominates all available assets on a
   stand-alone Sharpe-ratio basis.

2. **Short-selling is the key enabler**: Allowing even limited short positions
   (weights as low as −0.5) unlocks meaningful diversification, because it permits
   the endowment to *underweight* assets that have unfavourable return/risk
   profiles after accounting for correlation with the market.

3. **Among dimensions of improvement**, the **shorting dimension** is most
   responsible for the gains, not the direction of tilt per se.  Adding small-cap
   portfolios to the universe (Scenario 2 → 4) is beneficial primarily because it
   gives the endowment *more assets to short*, not more assets to hold long.

4. **Small-cap concerns are partially vindicated**: Committee members worried
   about small-cap risk are correct that small-cap portfolios have high standalone
   risk and low Sharpe ratios.  Their inclusion in a long-only portfolio adds no
   value.  However, in a short-enabled universe, they do contribute—as short
   candidates whose weak risk-adjusted returns can be harvested.
